In [ ]:
# ============================================================
# CELL 1 — Install & Import
# ============================================================

!pip install scikit-learn pandas numpy joblib -q

import pandas as pd
import numpy as np
import json
import re
import joblib
import os
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("✅ All imports done")

✅ All imports done


In [ ]:
# ============================================================
# CELL 2 — Load Data
# Upload both files first using the Colab files panel
# ============================================================

# Load Coursera CSV
df = pd.read_csv("Coursera.csv")
print("✅ Coursera shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(2))

# Load Taxonomy JSON
with open("it_skill_taxonomy.json") as f:
    taxonomy = json.load(f)

print("\n✅ Taxonomy categories:", list(taxonomy["categories"].keys()))
print("Total skills in taxonomy:", sum(len(v["skills"]) for v in taxonomy["categories"].values()))

✅ Coursera shape: (3522, 7)
Columns: ['Course Name', 'University', 'Difficulty Level', 'Course Rating', 'Course URL', 'Course Description', 'Skills']
                                         Course Name  \
0  Write A Feature Length Screenplay For Film Or ...   
1  Business Strategy: Business Model Canvas Analy...   

                  University Difficulty Level Course Rating  \
0  Michigan State University         Beginner           4.8   
1   Coursera Project Network         Beginner           4.8   

                                          Course URL  \
0  https://www.coursera.org/learn/write-a-feature...   
1  https://www.coursera.org/learn/canvas-analysis...   

                                  Course Description  \
0  Write a Full Length Feature Film Script  In th...   
1  By the end of this guided project, you will be...   

                                              Skills  
0  Drama  Comedy  peering  screenwriting  film  D...  
1  Finance  business plan  persona (user ex

In [ ]:
# ============================================================
# CELL 3 — Clean Coursera Skills
# ============================================================

def clean_skills(raw_skills):
    """
    Input:  "python programming  pandas  numpy data-science"
    Output: ["python", "pandas", "numpy", "data science"]
    """
    # Remove category tags at the end (e.g. data-science data-analysis)
    raw = str(raw_skills).lower()

    # Split by 2+ spaces or comma
    parts = re.split(r',|  +', raw)

    cleaned = []
    for part in parts:
        part = part.strip()
        # Skip short noise and category slugs (no spaces, has hyphens = slug)
        if len(part) < 2:
            continue
        if re.match(r'^[a-z]+-[a-z]+(-[a-z]+)*$', part):
            continue  # skip slugs like "data-science"
        cleaned.append(part)

    return cleaned


# Apply cleaning
df["skills_list"] = df["Skills"].apply(clean_skills)

# Build single string for TF-IDF later
df["skills_clean"] = df["skills_list"].apply(lambda x: " ".join(x))

# Drop rows with no skills
df = df[df["skills_clean"].str.strip() != ""].reset_index(drop=True)

print("✅ Cleaned. Rows remaining:", len(df))
print("\nSample:")
print(df[["Course Name", "skills_list"]].head(3))

✅ Cleaned. Rows remaining: 3522

Sample:
                                         Course Name  \
0  Write A Feature Length Screenplay For Film Or ...   
1  Business Strategy: Business Model Canvas Analy...   
2                      Silicon Thin Film Solar Cells   

                                         skills_list  
0  [drama, comedy, peering, screenwriting, film, ...  
1  [finance, business plan, persona (user experie...  
2  [chemistry, physics, solar energy, film, lambd...  


In [ ]:
# ============================================================
# CELL 4 — Map Each Course to a Taxonomy Category (Fixed)
# ============================================================

skill_to_category = {}
for category, cat_data in taxonomy["categories"].items():
    for skill_name, skill_data in cat_data["skills"].items():
        skill_to_category[skill_name.lower()] = category
        for alias in skill_data["aliases"]:
            skill_to_category[alias.lower()] = category

def assign_category(skills_list):
    votes = Counter()
    for skill in skills_list:
        skill_lower = skill.strip().lower()
        # Direct match only — no partial match (was causing false positives)
        if skill_lower in skill_to_category:
            votes[skill_to_category[skill_lower]] += 2  # stronger signal
        else:
            # Only match if skill is a substring of a known key (not reverse)
            for key, cat in skill_to_category.items():
                if skill_lower == key:
                    votes[cat] += 1
    if votes:
        return votes.most_common(1)[0][0]
    return "Other"

df["category"] = df["skills_list"].apply(assign_category)

print("✅ Category distribution:")
print(df["category"].value_counts())

df_it = df[df["category"] != "Other"].reset_index(drop=True)
print(f"\n✅ IT courses kept: {len(df_it)} / {len(df)}")

✅ Category distribution:
category
Other                 2591
Data Science & ML      627
Cloud & DevOps          93
Web Development         91
Cybersecurity           67
Mobile Development      38
Databases               15
Name: count, dtype: int64

✅ IT courses kept: 931 / 3522


In [ ]:
# ============================================================
# CELL 4.5 — Fix Data Quality Issues
# ============================================================

# Fix 1: Remove duplicates
before = len(df_it)
df_it = df_it.drop_duplicates(subset=["Course URL"]).reset_index(drop=True)
print(f"✅ Duplicates removed: {before - len(df_it)} | Rows now: {len(df_it)}")

# Fix 2: Fill missing ratings with category mean
df_it["Course Rating"] = pd.to_numeric(df_it["Course Rating"], errors="coerce")
df_it["Course Rating"] = df_it.groupby("category")["Course Rating"].transform(
    lambda x: x.fillna(x.mean())
)
print(f"✅ Rating nulls filled: {df_it['Course Rating'].isnull().sum()} remaining")

# Fix 3: Merge Databases into Cloud & DevOps (too few samples)
df_it["category"] = df_it["category"].replace("Databases", "Cloud & DevOps")
print(f"✅ Databases merged into Cloud & DevOps")

# Fix 4: Normalize difficulty levels
diff_map = {"Conversant": "Intermediate", "Not Calibrated": "Beginner"}
df_it["Difficulty Level"] = df_it["Difficulty Level"].replace(diff_map)
print(f"✅ Difficulty levels normalized")

# Final check
print(f"\nFinal class distribution:")
print(df_it["category"].value_counts())
print(f"\nDifficulty levels:")
print(df_it["Difficulty Level"].value_counts())
print(f"\nTotal clean rows: {len(df_it)}")

✅ Duplicates removed: 58 | Rows now: 873
✅ Rating nulls filled: 0 remaining
✅ Databases merged into Cloud & DevOps
✅ Difficulty levels normalized

Final class distribution:
category
Data Science & ML     592
Cloud & DevOps         95
Web Development        90
Cybersecurity          58
Mobile Development     38
Name: count, dtype: int64

Difficulty levels:
Difficulty Level
Beginner        460
Advanced        269
Intermediate    144
Name: count, dtype: int64

Total clean rows: 873


In [ ]:
# ============================================================
# CELL 5 — Train TF-IDF + Category Classifier (Fixed)
# ============================================================

# Check class distribution first
print("Class distribution:")
print(pd.Series(le.inverse_transform(y)).value_counts())

# Encode labels
# Only keep classes with enough samples (min 10)
class_counts = pd.Series(df_it["category"]).value_counts()
valid_classes = class_counts[class_counts >= 10].index.tolist()
print(f"\nKeeping classes with >=10 samples: {valid_classes}")

df_model = df_it[df_it["category"].isin(valid_classes)].reset_index(drop=True)
print(f"Rows for training: {len(df_model)}")

# Re-encode
le2 = LabelEncoder()
y = le2.fit_transform(df_model["category"])

# TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=3000)
X = tfidf.fit_transform(df_model["skills_clean"])

# Train/test split — stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Logistic Regression with class_weight balanced
clf = LogisticRegression(max_iter=1000, C=5,
                          class_weight="balanced",
                          random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
present_labels = np.unique(np.concatenate([y_test, y_pred]))

print(f"\n✅ Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    labels=present_labels,
    target_names=le2.classes_[present_labels]
))

# Update le to le2 for downstream cells
le = le2

Class distribution:
Data Science & ML     592
Cloud & DevOps         95
Web Development        90
Cybersecurity          58
Mobile Development     38
Name: count, dtype: int64

Keeping classes with >=10 samples: ['Data Science & ML', 'Cloud & DevOps', 'Web Development', 'Cybersecurity', 'Mobile Development']
Rows for training: 873

✅ Accuracy: 0.8971 (89.71%)

Classification Report:
                    precision    recall  f1-score   support

    Cloud & DevOps       0.73      0.84      0.78        19
     Cybersecurity       0.69      0.92      0.79        12
 Data Science & ML       0.98      0.91      0.94       119
Mobile Development       0.86      0.86      0.86         7
   Web Development       0.80      0.89      0.84        18

          accuracy                           0.90       175
         macro avg       0.81      0.88      0.84       175
      weighted avg       0.91      0.90      0.90       175



In [ ]:
# ============================================================
# CELL 5.5 — Overfitting Check
# ============================================================

train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc  = accuracy_score(y_test,  clf.predict(X_test))

print(f"Train Accuracy : {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Test  Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"Gap            : {(train_acc - test_acc)*100:.2f}%")

if (train_acc - test_acc) < 0.05:
    print("✅ No overfitting — gap is under 5%")
elif (train_acc - test_acc) < 0.10:
    print("⚠️ Slight overfitting — gap is under 10%")
else:
    print("❌ Overfitting detected — gap over 10%")

Train Accuracy : 0.9742 (97.42%)
Test  Accuracy : 0.8971  (89.71%)
Gap            : 7.71%
⚠️ Slight overfitting — gap is under 10%


In [ ]:
# ============================================================
# CELL 5.6 — Fix Slight Overfitting (Tune C)
# ============================================================

from sklearn.model_selection import cross_val_score

# Try different C values
for c in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]:
    model = LogisticRegression(max_iter=1000, C=c,
                                class_weight="balanced",
                                random_state=42)
    model.fit(X_train, y_train)
    tr = accuracy_score(y_train, model.predict(X_train))
    te = accuracy_score(y_test,  model.predict(X_test))
    print(f"C={c:<5} | Train: {tr:.4f} | Test: {te:.4f} | Gap: {(tr-te)*100:.2f}%")

C=0.01  | Train: 0.8997 | Test: 0.8400 | Gap: 5.97%
C=0.1   | Train: 0.9097 | Test: 0.8686 | Gap: 4.12%
C=0.5   | Train: 0.9298 | Test: 0.8800 | Gap: 4.98%
C=1.0   | Train: 0.9499 | Test: 0.8914 | Gap: 5.84%
C=2.0   | Train: 0.9670 | Test: 0.8914 | Gap: 7.56%
C=5.0   | Train: 0.9742 | Test: 0.8971 | Gap: 7.71%


In [ ]:
# ============================================================
# CELL 6 — Course Recommender (Cosine Similarity)
# ============================================================

# Pre-compute TF-IDF matrix for all IT courses (for retrieval)
course_matrix = tfidf.transform(df_it["skills_clean"])


def recommend_courses(user_skills: list, top_n: int = 5):
    """
    Given a list of user skills (gap skills),
    return top_n most relevant Coursera courses.
    """
    query = " ".join(user_skills).lower()
    query_vec = tfidf.transform([query])

    scores = cosine_similarity(query_vec, course_matrix).flatten()
    top_idx = scores.argsort()[::-1][:top_n]

    results = []
    for idx in top_idx:
        row = df_it.iloc[idx]
        results.append({
            "course_name": row["Course Name"],
            "difficulty": row["Difficulty Level"],
            "rating": row["Course Rating"],
            "url": row["Course URL"],
            "category": row["category"],
            "score": round(float(scores[idx]), 4)
        })
    return results


# ---- Test it ----
test_skills = ["machine learning", "python", "deep learning"]
print(f"🔍 Query skills: {test_skills}\n")
recs = recommend_courses(test_skills, top_n=5)
for i, r in enumerate(recs, 1):
    print(f"{i}. {r['course_name']}")
    print(f"   Category: {r['category']} | Level: {r['difficulty']} | Rating: {r['rating']}")
    print(f"   URL: {r['url']}")
    print(f"   Similarity Score: {r['score']}\n")

🔍 Query skills: ['machine learning', 'python', 'deep learning']

1. Predicting House Prices with Regression using TensorFlow
   Category: Data Science & ML | Level: Beginner | Rating: 4.5
   URL: https://www.coursera.org/learn/tensorflow-beginner-predicting-house-prices-regression
   Similarity Score: 0.4527

2. AI For Medical Treatment
   Category: Data Science & ML | Level: Beginner | Rating: 4.6
   URL: https://www.coursera.org/learn/ai-for-medical-treatment
   Similarity Score: 0.3979

3. Data for Machine Learning
   Category: Data Science & ML | Level: Beginner | Rating: 4.4
   URL: https://www.coursera.org/learn/data-machine-learning
   Similarity Score: 0.3867

4. Introduction to Deep Learning
   Category: Data Science & ML | Level: Advanced | Rating: 4.4
   URL: https://www.coursera.org/learn/intro-to-deep-learning
   Similarity Score: 0.3818

5. Introduction to Applied Machine Learning
   Category: Data Science & ML | Level: Intermediate | Rating: 4.7
   URL: https://www.cours

In [ ]:
# ============================================================
# CELL 7 — Skill Gap Analyzer (uses taxonomy JSON)
# ============================================================

def normalize_skill(skill: str) -> str:
    """Lowercase and strip a skill name."""
    return skill.lower().strip()


def find_skill_in_taxonomy(skill: str):
    """Return (canonical_name, category, skill_data) or None."""
    s = normalize_skill(skill)
    for category, cat_data in taxonomy["categories"].items():
        for skill_name, skill_data in cat_data["skills"].items():
            if s == skill_name.lower() or s in [a.lower() for a in skill_data["aliases"]]:
                return skill_name, category, skill_data
    return None


def analyze_skill_gap(user_skills: list):
    """
    Input : ["Python", "React", "SQL"]
    Output: {
        known_skills, known_categories,
        gap_skills (trending skills user is missing),
        recommendations (courses for gap skills)
    }
    """
    user_normalized = [normalize_skill(s) for s in user_skills]

    # Step 1: Find what categories the user belongs to
    user_category_votes = Counter()
    matched_skills = {}

    for skill in user_normalized:
        result = find_skill_in_taxonomy(skill)
        if result:
            canonical, category, data = result
            user_category_votes[category] += data["demand_score"]
            matched_skills[canonical] = {"category": category, **data}

    # Step 2: Determine top 2 user categories
    top_categories = [cat for cat, _ in user_category_votes.most_common(2)]

    # Step 3: Find gap = trending skills in those categories user doesn't have
    gap_skills = []
    for category in top_categories:
        cat_skills = taxonomy["categories"][category]["skills"]
        for skill_name, skill_data in cat_skills.items():
            if skill_data["trending"] and skill_name not in matched_skills:
                gap_skills.append({
                    "skill": skill_name,
                    "category": category,
                    "demand_score": skill_data["demand_score"],
                    "level": skill_data["level"]
                })

    # Sort gap by demand score
    gap_skills = sorted(gap_skills, key=lambda x: x["demand_score"], reverse=True)

    # Step 4: Recommend courses for top 5 gap skills
    gap_skill_names = [g["skill"] for g in gap_skills[:5]]
    courses = recommend_courses(gap_skill_names, top_n=5)

    return {
        "known_skills": list(matched_skills.keys()),
        "top_categories": top_categories,
        "gap_skills": gap_skills[:8],
        "recommended_courses": courses
    }


# ---- Test it ----
user = ["Python", "SQL", "Pandas", "React"]
result = analyze_skill_gap(user)

print(f"👤 User Skills   : {user}")
print(f"📂 Top Categories: {result['top_categories']}")
print(f"\n✅ Known Skills  : {result['known_skills']}")
print(f"\n❌ Skill Gaps ({len(result['gap_skills'])}):")
for g in result["gap_skills"]:
    print(f"   - {g['skill']} | {g['category']} | Demand: {g['demand_score']} | Level: {g['level']}")

print(f"\n📚 Recommended Courses:")
for i, c in enumerate(result["recommended_courses"], 1):
    print(f"   {i}. {c['course_name']} ({c['difficulty']}) ⭐{c['rating']}")

👤 User Skills   : ['Python', 'SQL', 'Pandas', 'React']
📂 Top Categories: ['Data Science & ML', 'Web Development']

✅ Known Skills  : ['Python', 'SQL', 'Pandas', 'React']

❌ Skill Gaps (8):
   - Generative AI | Data Science & ML | Demand: 98 | Level: Advanced
   - JavaScript | Web Development | Demand: 98 | Level: Beginner
   - Machine Learning | Data Science & ML | Demand: 95 | Level: Intermediate
   - Data Analysis | Data Science & ML | Demand: 92 | Level: Intermediate
   - TypeScript | Web Development | Demand: 92 | Level: Intermediate
   - REST API | Web Development | Demand: 92 | Level: Intermediate
   - NLP | Data Science & ML | Demand: 91 | Level: Advanced
   - Deep Learning | Data Science & ML | Demand: 90 | Level: Advanced

📚 Recommended Courses:
   1. Exploratory Data Analysis (Beginner) ⭐4.3
   2. Analyze Box Office Data with Seaborn and Python (Beginner) ⭐4.5
   3. Building AI Applications with Watson APIs (Beginner) ⭐3.8
   4. Applied Machine Learning in Python (Advanced) ⭐

In [ ]:
# ============================================================
# CELL 8 — Save & Download Models
# ============================================================

import os, json, joblib, shutil

os.makedirs('skillink_dev07_models', exist_ok=True)

# Save ML models
joblib.dump(tfidf, 'skillink_dev07_models/tfidf_dev07.joblib')
joblib.dump(clf,   'skillink_dev07_models/clf_category_dev07.joblib')
joblib.dump(le,    'skillink_dev07_models/label_encoder_dev07.joblib')

# Save cleaned courses CSV
df_it[["Course Name", "Difficulty Level", "Course Rating",
       "Course URL", "category", "skills_clean"]].to_csv(
    'skillink_dev07_models/courses_clean.csv', index=False
)

# Copy taxonomy
shutil.copy('it_skill_taxonomy.json', 'skillink_dev07_models/it_skill_taxonomy.json')

print("✅ Saved:")
for f in os.listdir('skillink_dev07_models'):
    size = os.path.getsize(f'skillink_dev07_models/{f}') / 1024
    print(f"   {f}  ({size:.1f} KB)")

# Zip & download
shutil.make_archive('skillink_dev07_models', 'zip', 'skillink_dev07_models')

from google.colab import files
files.download('skillink_dev07_models.zip')
print("\n📦 Download started!")

✅ Saved:
   it_skill_taxonomy.json  (7.9 KB)
   courses_clean.csv  (273.5 KB)
   tfidf_dev07.joblib  (126.8 KB)
   label_encoder_dev07.joblib  (0.6 KB)
   clf_category_dev07.joblib  (118.1 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📦 Download started!
